In [ ]:
import asyncio
from openai import OpenAI
import nest_asyncio
nest_asyncio.apply()

# 1. Standard OpenAI client pointed at your local llama.cpp server.
client = OpenAI(base_url="http://localhost:8080/v1", api_key="local-no-key-required")

# ---------------------------------------------------------------------------
# Model discovery + prompt templating
# ---------------------------------------------------------------------------
# We drive the raw /v1/completions endpoint so we can inject a partial assistant
# turn and swap grammars mid-stream. That means WE own the prompt scaffolding,
# which is model-family specific (harmony, chatml, llama3, ...). So: ask the
# server which model is loaded, map it to a template family, and build the prompt
# for that family.

def get_current_model_name() -> str:
    # llama.cpp exposes the loaded model via the OpenAI-compatible /v1/models.
    return client.models.list().data[0].id

def get_template_family(model_name: str) -> str:
    """Best-effort map from a model id to its chat-template family."""
    n = model_name.lower()
    if "gpt-oss" in n or "harmony" in n:
        return "harmony"
    if "llama-3" in n or "llama3" in n:
        return "llama3"
    if "gemma" in n:
        return "gemma"
    if "phi" in n:
        return "phi"
    if any(k in n for k in ("mistral", "mixtral", "llama-2", "llama2")):
        return "mistral"
    if any(k in n for k in ("qwen", "yi", "chatml", "hermes")):
        return "chatml"
    return "chatml"  # sensible, widely-compatible default

def build_prompt(user_prompt: str, template_family: str, assistant_so_far: str = "") -> str:
    """Render a single user turn + an OPEN assistant turn for the given family.

    The assistant turn is deliberately left unterminated (ends right where the
    assistant's content begins) and pre-filled with `assistant_so_far`, so raw
    completion resumes generating the assistant message -- that's what makes the
    grammar-swap / continuation trick work.
    """
    if template_family == "harmony":
        # gpt-oss. Prime the `final` channel to skip the long analysis/reasoning turn.
        return (
            "<|start|>user<|message|>" + user_prompt + "<|end|>"
            "<|start|>assistant<|channel|>final<|message|>" + assistant_so_far
        )
    if template_family == "chatml":
        return (
            "<|im_start|>user\n" + user_prompt + "<|im_end|>\n"
            "<|im_start|>assistant\n" + assistant_so_far
        )
    if template_family == "llama3":
        return (
            "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"
            + user_prompt + "<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n" + assistant_so_far
        )
    if template_family == "gemma":
        # Note: Gemma's assistant role is called "model" and has no system role.
        return (
            "<start_of_turn>user\n" + user_prompt + "<end_of_turn>\n"
            "<start_of_turn>model\n" + assistant_so_far
        )
    if template_family == "phi":
        return (
            "<|user|>\n" + user_prompt + "<|end|>\n"
            "<|assistant|>\n" + assistant_so_far
        )
    if template_family == "mistral":
        return "<s>[INST] " + user_prompt + " [/INST]" + assistant_so_far
    raise ValueError(f"Unknown template family: {template_family!r}")

# ANSI escapes: make the FIRST character of each token bright/bold so you can see
# token boundaries live, while the rest of the token stays normal. No "|" clutter.
BRIGHT, RESET = "\033[1;97m", "\033[0m"   # bold bright-white, then reset

def colorize_token(tok: str) -> str:
    if not tok:
        return tok
    return f"{BRIGHT}{tok[0]}{RESET}{tok[1:]}"

MODEL_NAME = get_current_model_name()
TEMPLATE_FAMILY = get_template_family(MODEL_NAME)
print(f"Loaded model : {MODEL_NAME}")
print(f"Template     : {TEMPLATE_FAMILY}\n")


async def open_ai_dynamic_stream(user_prompt: str):
    assistant_text = ""                  # everything the model has committed so far
    current_grammar = 'root ::= [^\n]+'  # Step 1: free text up until a newline
    generation_active = True
    swapped = False                      # one-shot: has the grammar been swapped yet?
    token_index = 0                      # counter so we can watch each token arrive
    print("AI: ", end="", flush=True)

    while generation_active:
        # Raw text completion. The full prompt (incl. what was generated so far) is
        # rebuilt each cycle; llama.cpp's prompt cache makes the resume instant.
        stream = client.completions.create(
            model=MODEL_NAME,
            prompt=build_prompt(user_prompt, TEMPLATE_FAMILY, assistant_text),
            stream=True,
            max_tokens=512,
            extra_body={
                "grammar": current_grammar  # Enforces our grammar rule on llama.cpp!
            }
        )

        cycle_text = ""

        # Stream the tokens out chunk by chunk
        for chunk in stream:
            token_text = chunk.choices[0].text or ""   # note: .text, not .delta.content
            if token_text == "":
                continue  # skip empty/keep-alive chunks
            cycle_text += token_text
            token_index += 1

            # Print the token inline; its first character is brightened so you can
            # watch tokens arrive and see their boundaries without any delimiter.
            print(colorize_token(token_text), end="", flush=True)
            await asyncio.sleep(0)  # yield control in async environment

            # --- EVALUATE STREAM CONTENTS ON THE FLY ---
            # Intercept ONCE, as soon as the model emits our trigger "Action Required:".
            # The `not swapped` guard makes this a one-shot: without it the swapped
            # grammar (" REBOOT" | " SHUTDOWN") emits its shared leading space first,
            # which still matches the trigger, and we'd re-swap forever -> infinite loop.
            if not swapped and "Action Required:" in (assistant_text + cycle_text):
                # 1. Commit what it generated during this cycle
                assistant_text += cycle_text

                # 2. CHANGE THE GUIDE: mutate the grammar string. Now force llama.cpp
                #    to ONLY let the model output " REBOOT" or " SHUTDOWN".
                current_grammar = 'root ::= " REBOOT" | " SHUTDOWN"'
                swapped = True
                print(f"\n>>> Trigger detected -> swapping grammar to: {current_grammar}\nAI: ", end="", flush=True)

                # 3. Break to force the grammar swap; the while-loop re-issues the request.
                break
        else:
            # Loop finished naturally (stop token / max_tokens): commit and wrap up.
            # After the swap this is where we land once " REBOOT"/" SHUTDOWN" completes.
            assistant_text += cycle_text
            generation_active = False

        # If we broke out early to swap grammars, the while-loop continues and resends
        # the prompt (now ending with the text generated so far) under the new grammar.
        # Prompt caching resumes from the exact last token instantly.

    print(f"\n\n--- Done. {token_index} tokens total. ---")
    print("Final assistant text:", repr(assistant_text))

# Run the async stream
asyncio.run(open_ai_dynamic_stream("The server is overheating. What should I do? Please use the prefix 'Action Required:'"))
